# Demo — Entrena tu propio BPE

> Demo interactiva que acompaña a [Grandes Modelos de Lenguaje e
> Inferencia](../bloque1/01b-llms.qmd#sec-tokenizacion-detalle). Todo
> corre en tu navegador: no necesitas clave de API.

El algoritmo **Byte Pair Encoding (BPE)** empieza con un vocabulario de
caracteres y, en cada paso, fusiona el par de símbolos adyacentes más
frecuente. Cada fusión queda registrada como una regla que luego se
aplica en ese mismo orden.

Prueba a:

- Subir el número de fusiones y ver cómo aparecen subpalabras como `low`
  o `est`.
- Cambiar el corpus: si una palabra no aparece en el entrenamiento, BPE
  la descompone en piezas más pequeñas.
- Tokenizar palabras que no estén en el corpus, como `unhappiness`.

``` shinylive-python
#| '!! shinylive warning !!': |
#|   shinylive does not work in self-contained HTML documents.
#|   Please set `embed-resources: false` in your metadata.
#| standalone: true
#| viewerHeight: 840

import html
from collections import Counter

from shiny import reactive
from shiny.express import input, ui, render

CORPUS = "low lower lowest newer newer newest wider widest new new low low"
END = "</w>"
PALETTE = ["#dbeafe", "#fde68a", "#dcfce7", "#fbcfe8", "#e9d5ff", "#fed7aa", "#ccfbf1", "#e5e7eb"]
ORANGE = "#e65100"

ui.tags.style("""
body{font-family:system-ui,-apple-system,'Segoe UI',sans-serif}
.card{border:1px solid #e5e7eb;border-radius:10px;padding:10px 14px;margin:10px 0;background:#fff}
.muted{color:#6b7280;font-size:.85em}
.tok{display:inline-block;color:#111;padding:1px 6px;margin:1px;border-radius:4px;
  font-family:ui-monospace,Menlo,monospace;border:1.5px solid transparent}
.tok.char{border:1.5px dashed #9ca3af;background:#fff!important}
.tok.fresh{border-color:#e65100;box-shadow:0 0 0 2px #e6510033}
.word{display:inline-flex;align-items:center;margin:3px 14px 3px 0;white-space:nowrap}
.freq{font-size:.75em;color:#6b7280;margin-left:3px}
.rules{display:flex;flex-wrap:wrap;gap:6px;margin:6px 0}
.rule{border-radius:8px;padding:3px 9px;font-size:.9em;white-space:nowrap;color:#111;border:1.5px solid transparent}
.rule.fresh{border-color:#e65100}
.rule code{background:none;color:#111}
@keyframes pop{from{transform:scale(.6);opacity:0}to{transform:scale(1);opacity:1}}
.fresh{animation:pop .4s ease-out both}
""")


def color(tok):
    """Mismo token, mismo color en toda la página."""
    return PALETTE[sum(map(ord, tok)) % len(PALETTE)]


def show(s):
    return html.escape(s.replace(END, "_"))


def merge_word(word, pair):
    """Aplica una fusión a una secuencia de símbolos."""
    out, i = [], 0
    while i < len(word):
        if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
            out.append(word[i] + word[i + 1])
            i += 2
        else:
            out.append(word[i])
            i += 1
    return tuple(out)


def train_bpe(text, n_merges):
    """Devuelve las reglas en orden, su frecuencia y el corpus ya troceado."""
    words = Counter()
    for w in text.lower().split():
        words[tuple(w) + (END,)] += 1

    merges, freqs = [], []
    for _ in range(n_merges):
        pairs = Counter()
        for word, freq in words.items():
            for pair in zip(word, word[1:]):
                pairs[pair] += freq
        if not pairs:
            break
        best = max(pairs, key=lambda p: (pairs[p], p))
        merges.append(best)
        freqs.append(pairs[best])
        new_words = Counter()
        for word, freq in words.items():
            new_words[merge_word(word, best)] += freq
        words = new_words
    return merges, freqs, words


def tokenize(word, merges):
    """Trocea una palabra aplicando las reglas en orden."""
    toks = tuple(word) + (END,)
    for pair in merges:
        toks = merge_word(toks, pair)
    return list(toks)


def pill(t, merges, fresh):
    origin = {a + b: k + 1 for k, (a, b) in enumerate(merges)}
    if t in origin:
        cls, tip = "tok" + (" fresh" if t == fresh else ""), f"creado por la regla {origin[t]}"
    else:
        cls, tip = "tok char", "carácter suelto: ninguna regla lo ha unido"
    return f"<span class='{cls}' style='background:{color(t)}' title='{tip}'>{show(t)}</span>"


@reactive.calc
def model():
    return train_bpe(input.corpus(), input.n_merges())


ui.input_text_area("corpus", "Corpus de entrenamiento", CORPUS, rows=2, width="100%")
ui.input_slider(
    "n_merges", "Número de fusiones (pulsa ▶ para verlas una a una)", 0, 40, 12, width="100%",
    animate=ui.AnimationOptions(interval=900, loop=False),
)


@render.ui
def corpus_out():
    merges, freqs, words = model()
    fresh = merges[-1][0] + merges[-1][1] if merges else None
    n_toks = sum(len(w) * f for w, f in words.items())
    n_chars = sum((len("".join(w).replace(END, "")) + 1) * f for w, f in words.items())
    items = "".join(
        f"<span class='word'>{''.join(pill(t, merges, fresh) for t in w)}<span class='freq'>×{f}</span></span>"
        for w, f in sorted(words.items(), key=lambda kv: -kv[1])
    )
    last = ""
    if merges:
        a, b = merges[-1]
        last = (
            f" · última regla: <code>{show(a)} + {show(b)}</code> "
            f"(el par aparecía {freqs[-1]} veces)"
        )
    return ui.HTML(
        f"<div class='card'><b>El corpus, troceado con las reglas actuales</b>"
        f"<div class='muted'>{n_chars} símbolos iniciales → <b>{n_toks}</b> tokens{last}</div>"
        f"<div style='margin-top:6px'>{items}</div></div>"
    )


@render.ui
def merges_out():
    merges, freqs, _ = model()
    if not merges:
        return ui.HTML("<div class='card'><b>Reglas aprendidas</b><p class='muted'>Sin fusiones: solo caracteres.</p></div>")
    cards = "".join(
        f"<div class='rule{' fresh' if k == len(merges) - 1 else ''}' style='background:{color(a + b)}'>"
        f"<span style='opacity:.6'>{k + 1}.</span> <code>{show(a)} + {show(b)}</code> → "
        f"<b><code>{show(a + b)}</code></b></div>"
        for k, (a, b) in enumerate(merges)
    )
    return ui.HTML(f"<div class='card'><b>Reglas aprendidas (en orden)</b><div class='rules'>{cards}</div></div>")


ui.input_text("probe", "Texto a tokenizar", "lowest newest unhappiness", width="100%")


@render.ui
def tokens_out():
    merges, _, _ = model()
    words = input.probe().lower().split()
    chunks, n_tokens = [], 0
    for w in words:
        toks = tokenize(w, merges)
        n_tokens += len(toks)
        chunks.append(f"<span class='word'>{''.join(pill(t, merges, None) for t in toks)}</span>")
    return ui.HTML(
        f"<div class='card'><b>Tokenización</b> "
        f"<span class='muted'>{len(words)} palabras → <b>{n_tokens}</b> tokens</span>"
        f"<div style='margin-top:6px'>{''.join(chunks)}</div>"
        f"<p class='muted' style='margin:6px 0 0'>Mismo color = mismo token. "
        f"<span class='tok char'>x</span> = carácter suelto que ninguna regla ha unido. "
        f"Pasa el ratón por un token para ver qué regla lo creó.</p></div>"
    )
```

## Y un tokenizador real, ya entrenado

Lo de arriba es un BPE de juguete, entrenado sobre doce palabras. Ahora
escribe cualquier texto y mira cómo lo trocea el tokenizador de
**Qwen3-0.6B**, con un vocabulario de más de 150 000 tokens. Corre en tu
navegador: se descarga una vez (unos 11 MB) y después tokeniza al
instante, sin servidor.

## Qué observar

- Pon el deslizador en 0 y pulsa **▶**: verás las fusiones una a una. La
  pieza recién creada se marca en naranja en el corpus, y el número de
  tokens baja con cada regla.
- Los tokens con borde discontinuo son caracteres sueltos que ninguna
  regla ha unido: son la señal de que el tokenizador no conocía esa
  parte de la palabra.
- El símbolo `_` marca el final de palabra (`</w>` en la notación del
  paper de Sennrich et al.). Permite distinguir `est` al final de una
  palabra de `est` dentro de otra.
- Una palabra rara como `unhappiness` se parte en piezas que el modelo
  ya conoce. Por eso BPE no tiene el problema de palabras fuera de
  vocabulario.
- Compara el BPE de juguete con Qwen3: el tokenizador real aprendió
  sobre una cantidad enorme de texto, por eso frases comunes como *«Los
  transformers procesan»* salen en muy pocos tokens.
- Pulsa los ejemplos **Español** e **Inglés** de Qwen3: dicen lo mismo,
  pero en español hacen falta muchos más tokens. El vocabulario se
  aprendió con más texto en inglés, y como se paga y se limita por
  tokens, el mismo contenido en español cuesta más y ocupa más contexto.
- El símbolo `·` marca un espacio al inicio de la palabra. Qwen3 lo
  incluye dentro del token, como hace la mayoría de tokenizadores
  modernos.

> **Limitación**
>
> Esta es una implementación didáctica sobre un corpus mínimo. Los
> tokenizadores reales (GPT, Llama, Qwen) entrenan con miles de millones
> de palabras, manejan bytes en lugar de caracteres y tienen decenas de
> miles de tokens.